In [24]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/last-form-images/last_form_images/36_hole_aug2.jpg
/kaggle/input/last-form-images/last_form_images/11_hole_aug2.jpg
/kaggle/input/last-form-images/last_form_images/33_hole_aug1.jpg
/kaggle/input/last-form-images/last_form_images/15_vertical_aug1.jpg
/kaggle/input/last-form-images/last_form_images/27_horizontal_aug4.jpg
/kaggle/input/last-form-images/last_form_images/12_horizontal_aug2.jpg
/kaggle/input/last-form-images/last_form_images/16_vertical_aug4.jpg
/kaggle/input/last-form-images/last_form_images/10_hole_aug2.jpg
/kaggle/input/last-form-images/last_form_images/22_horizontal_aug2.jpg
/kaggle/input/last-form-images/last_form_images/21_horizontal_aug1.jpg
/kaggle/input/last-form-images/last_form_images/14_horizontal_aug0.jpg
/kaggle/input/last-form-images/last_form_images/40_hole_aug3.jpg
/kaggle/input/last-form-images/last_form_images/15_horizontal_aug0.jpg
/kaggle/input/last-form-images/last_form_images/29_hole_aug2.jpg
/kaggle/input/last-form-images/last_form_image

In [40]:
import os
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from sklearn.metrics import f1_score
import torchvision.transforms as transforms
import torch.nn.functional as F
import random
#import torch.nn.functional as F_vision
import torchvision.transforms.functional as TF

In [41]:
# ---------------------------
# 1. Dataset
# ---------------------------
class SegmentationDataset(Dataset):
    def __init__(self, images_dir, masks_dir, img_transform=None, mask_transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.img_transform = img_transform
        self.mask_transform = mask_transform

        self.images = sorted(os.listdir(images_dir))
        self.masks = sorted(os.listdir(masks_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.images[idx])
        mask_path = os.path.join(self.masks_dir, self.masks[idx])
    
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")
    
        # Görüntüye Resize
        image = TF.resize(image, TARGET_SIZE, interpolation=TF.InterpolationMode.BILINEAR)
        
        # Maskeye Resize (Nearest kullanmak maskeler için daha güvenlidir!)
        mask = TF.resize(mask, TARGET_SIZE, interpolation=TF.InterpolationMode.NEAREST)
        
        # 2. Rastgele Yatay Çevirme (Horizontal Flip)
        if random.random() < 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)
            
        # 3. Rastgele Dikey Çevirme (Vertical Flip)
        if random.random() < 0.5:
            image = TF.vflip(image)
            mask = TF.vflip(mask)
            
        # ------------------------------------------------------------------
        
        # 4. Geriye Kalan Dönüşümler
        
        # Görüntü ToTensor
        image = TF.to_tensor(image)
        
        # Maske işleme
        mask = np.array(mask) # PIL'den NumPy'a
        mask = (mask > 128).astype(np.uint8) # Binary mask
        
        # Tensöre çevir ve tek kanal ekle
        mask = torch.from_numpy(mask).long()
        mask = mask.unsqueeze(0)
        
        return image, mask

In [42]:
# ---------------------------
# 2. Transform ve DataLoader
# ---------------------------
images_dir = "/kaggle/input/last-form-images/last_form_images"
masks_dir = "/kaggle/input/last-form-masks/last_form_masks"

TARGET_SIZE = (360, 640)

img_transform = transforms.Compose([
    transforms.ToTensor(), # Görüntü için standart son adım
])

mask_transform = None

dataset = SegmentationDataset(images_dir, masks_dir, img_transform, mask_transform)

dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Validation split
val_size = int(0.2 * len(dataset))
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)


In [43]:
imgs, masks = next(iter(DataLoader(dataset, batch_size=4)))
print(imgs.shape, masks.shape)


torch.Size([4, 3, 360, 640]) torch.Size([4, 1, 360, 640])


In [45]:
# =========================
# Dice Loss
# =========================
def dice_loss(pred, target, smooth=1e-6):
    pred = pred.contiguous()
    target = target.contiguous()

    intersection = (pred * target).sum()
    dice = (2. * intersection + smooth) / \
           (pred.sum() + target.sum() + smooth)

    return 1 - dice

# =========================
# Focal Loss
# =========================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # inputs: Model çıktısı (sigmoid sonrası), targets: Gerçek maske (float)
        BCE_loss = F.binary_cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1-pt)**self.gamma * BCE_loss

        if self.reduction == 'mean':
            return F_loss.mean()
        elif self.reduction == 'sum':
            return F_loss.sum()
        else:
            return F_loss


In [55]:
# =========================
# U-Net
# =========================
class FlexibleUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, filters=[64, 128, 256, 512]):
        super(FlexibleUNet, self).__init__()
        
        # CBR: Çift Evrişim Bloğu
        def CBR(in_ch, out_ch):
            return nn.Sequential(
                # Birinci Konvolüsyon Bloğu
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                # İkinci Konvolüsyon Bloğu
                nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True)
            )

        self.num_levels = len(filters)

        # Encoder
        self.encoders = nn.ModuleList()
        prev_ch = in_channels
        for f in filters:
            self.encoders.append(CBR(prev_ch, f))
            prev_ch = f
        self.pool = nn.MaxPool2d(2)

        # Decoder
        self.decoders = nn.ModuleList()
        reversed_filters = list(reversed(filters))
        for i in range(len(reversed_filters) - 1):
            # Geri kalan seviyeler için U-Net'in klasik atlama bağlantısı (skip connection)
            self.decoders.append(CBR(reversed_filters[i] + reversed_filters[i+1], reversed_filters[i+1]))

        # Output
        self.final = nn.Conv2d(filters[0], out_channels, kernel_size=1)

    def forward(self, x):
        enc_features = []

        # Encoder yolu
        out = x
        for enc in self.encoders:
            out = enc(out)
            enc_features.append(out)
            out = self.pool(out)

        # Decoder yolu
        for i, dec in enumerate(self.decoders):
            skip = enc_features[-(i+2)]
            out = F.interpolate(out, size=skip.shape[2:], mode='bilinear', align_corners=True)
            out = torch.cat([out, skip], dim=1)
            out = dec(out)

        out = self.final(out)
        return torch.sigmoid(out)


# =========================
# Training Setup
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"

model = FlexibleUNet().to(device)
focal_criterion = FocalLoss(alpha=0.75, gamma=5.0).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)


def dice_score(y_true, y_pred, smooth=1e-6):
    y_true = y_true.view(-1)
    y_pred = y_pred.view(-1)
    intersection = (y_true * y_pred).sum()
    return (2. * intersection + smooth) / (y_true.sum() + y_pred.sum() + smooth)


epochs = 30
best_val_loss = float("inf")


# =========================
# Training Loop
# =========================
from sklearn.metrics import f1_score

for epoch in range(epochs):

    # Train
    model.train()
    train_loss = 0

    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)

        optimizer.zero_grad()

        outputs = model(imgs)

        # dice_loss fonksiyonunun tanımlı olduğunu varsayarız
        loss = focal_criterion(outputs, masks.float()) + dice_loss(outputs, masks.float())

        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    dice_total = 0
    f1_total = 0
    acc_total = 0

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)

            outputs = model(imgs)

            loss = focal_criterion(outputs, masks.float()) + dice_loss(outputs, masks.float())
            val_loss += loss.item()

            preds = (outputs > 0.5).float()

            dice_total += dice_score(masks, preds).item()

            f1_total += f1_score(
                masks.cpu().numpy().flatten(),
                preds.cpu().numpy().flatten()
            )

            acc_total += (preds == masks).float().mean().item()

    val_loss /= len(val_loader)
    dice_total /= len(val_loader)
    f1_total /= len(val_loader)
    acc_total /= len(val_loader)

    print(f"Epoch [{epoch+1}/{epochs}] | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Dice: {dice_total:.4f} | F1: {f1_total:.4f} | Acc: {acc_total:.4f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_unet_640x360.pth")
        print("Saved Best Model!")

Epoch [1/30] | Train Loss: 0.9885 | Val Loss: 1.3878 | Dice: 0.0123 | F1: 0.0123 | Acc: 0.6207
Saved Best Model!
Epoch [2/30] | Train Loss: 0.9828 | Val Loss: 4.4831 | Dice: 0.0215 | F1: 0.0215 | Acc: 0.7739
Epoch [3/30] | Train Loss: 0.9750 | Val Loss: 1.5459 | Dice: 0.0259 | F1: 0.0259 | Acc: 0.8369
Epoch [4/30] | Train Loss: 0.9442 | Val Loss: 0.9970 | Dice: 0.0502 | F1: 0.0502 | Acc: 0.9778
Saved Best Model!
Epoch [5/30] | Train Loss: 0.8701 | Val Loss: 1.9981 | Dice: 0.0342 | F1: 0.0342 | Acc: 0.8301
Epoch [6/30] | Train Loss: 0.8374 | Val Loss: 1.1183 | Dice: 0.0784 | F1: 0.0784 | Acc: 0.9470
Epoch [7/30] | Train Loss: 0.8077 | Val Loss: 0.9106 | Dice: 0.1464 | F1: 0.1464 | Acc: 0.9913
Saved Best Model!
Epoch [8/30] | Train Loss: 0.7522 | Val Loss: 1.3444 | Dice: 0.0945 | F1: 0.0945 | Acc: 0.9055
Epoch [9/30] | Train Loss: 0.7388 | Val Loss: 0.9971 | Dice: 0.0780 | F1: 0.0780 | Acc: 0.9887
Epoch [10/30] | Train Loss: 0.7036 | Val Loss: 0.9371 | Dice: 0.1586 | F1: 0.1586 | Acc: 0.

KeyboardInterrupt: 

In [31]:
print(masks.shape, masks.min().item(), masks.max().item())


torch.Size([4, 1, 360, 640]) 0 1


In [ ]:
print(masks.dtype)
print(torch.unique(masks))
